In [5]:
import polars as pl

# Import power plants
plants_df = pl.read_csv('Distances - Power Plants.csv')

# Import towns
towns_df = pl.read_csv('Distances - Towns.csv')

In [6]:
import math

# Formula to get the distance between two locations from the longitutude and latitutude in km
def haversine(lat1, long1, lat2, long2):
    # Convert degrees to radians
    lat1, long1, lat2, long2 = map(math.radians, [lat1, long1, lat2, long2])
    
    return 2 * 6371 * math.asin(
        math.sqrt(
            math.sin((lat2 - lat1) / 2)**2 + 
            math.cos(lat1) * math.cos(lat2) * math.sin((long2 - long1) / 2)**2
        )
    )

In [7]:
rows = [] # List of (plant, town, distance) tuples

# Iterate through power plants and town pairs
for plant_row in plants_df.iter_rows(named=True):
    for town_row in towns_df.iter_rows(named=True):
        distance = haversine(plant_row["Latitude"], plant_row["Longitude"], town_row["Latitude"], town_row["Longitude"])
        rows.append((plant_row["Power Plant"], town_row["Towns"], distance))

In [13]:
columns = ["power_plant", "town", "distance"]
rows_df = pl.DataFrame(rows, schema=columns, orient="row")
rows_df.head()

power_plant,town,distance
str,str,f64
"""Essential Power Newington""","""Manchester""",54.185814
"""Essential Power Newington""","""Nashua""",65.841638
"""Essential Power Newington""","""Concord""",66.074966
"""Essential Power Newington""","""Portsmouth""",4.924798
"""Essential Power Newington""","""Rochester""",26.186799


In [14]:
# Add ID's for power plants and towns
# Use the row index to match the order in the CSVs
plants_ids = plants_df.select(pl.col("Power Plant").alias("power_plant")).with_columns(power_plant_id=pl.int_range(1, pl.len() + 1))
towns_ids = towns_df.select(pl.col("Towns").alias("town")).with_columns(town_id=pl.int_range(1, pl.len() + 1))

rows_df = rows_df.join(plants_ids, on="power_plant", how="left")
rows_df = rows_df.join(towns_ids, on="town", how="left")

# Drop the old rank columns if they exist from before
if "power_plant_id_right" in rows_df.columns:
    rows_df = rows_df.drop(["power_plant_id_right", "town_id_right"])

rows_df.head()

power_plant,town,distance,power_plant_id,town_id
str,str,f64,i64,i64
"""Essential Power Newington""","""Manchester""",54.185814,1,1
"""Essential Power Newington""","""Nashua""",65.841638,1,2
"""Essential Power Newington""","""Concord""",66.074966,1,3
"""Essential Power Newington""","""Portsmouth""",4.924798,1,4
"""Essential Power Newington""","""Rochester""",26.186799,1,5


In [15]:
rows_df.write_csv("distance_pairs.csv")